# Objectif du problème

Classification binaire : prédire si un vol aura plus de 15 minutes de retard (ArrDel15).

In [ ]:
%matplotlib inline

In [ ]:
target = 'ArrDel15'

In [ ]:
# Importation des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Model Selection
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Métriques
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
    roc_curve,
    auc,
    precision_recall_curve,
)

def plot_confusion_matrix(y_true, y_pred, model_name):
    # 1. Plot compact
    plt.figure(figsize=(5, 4))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=["À l'heure", 'Retard'],
                yticklabels=["À l'heure", 'Retard'],
                cbar=False)
    plt.title(f'Matrice : {model_name}')
    plt.tight_layout()
    plt.show()

    # 2. Métriques condensées
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"\n--- {model_name.upper()} ---")
    print(f"Confusion : TN={tn} | FP={fp} | FN={fn} | TP={tp}")
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.2%}")
    print(f"Precision : {precision_score(y_true, y_pred, zero_division=0):.2%}")
    print(f"Recall    : {recall_score(y_true, y_pred, zero_division=0):.2%}")
    print(f"F1-Score  : {f1_score(y_true, y_pred, zero_division=0):.2%}")


## 1. Données : chargement et échantillonnage

In [ ]:
file_path = 'flight-delay-dataset-20182022/Combined_Flights_2021.parquet'

df = pd.read_parquet(file_path)

df = df.sample(n=100000, random_state=42).reset_index(drop=True)

print(f"Dataset (sample) chargé : {len(df):,} vols")

## 2. Exploration initiale (EDA rapide)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

### 2.1 Valeurs catégorielles clés

In [ ]:
df['Airline'].value_counts()

On observe des données manquantes dans plusieurs colonnes, principalement liées aux informations de vol effectives (retards, temps de taxi, etc.), ce qui est cohérent avec la présence de vols annulés ou détournés :

- **Départ** : `DepTime`, `DepDelay`, `DepDel15`, etc. (~1,8% de manquants).
- **Arrivée** : `ArrTime`, `ArrDelay`, `ArrDel15`, etc. (~2,1% de manquants).
- **Détails techniques** : `Tail_Number` (384 manquants) et `TaxiOut`/`TaxiIn`.
- **Informations de vol** : `AirTime` et `ActualElapsedTime`.

Ces valeurs devront être traitées (imputation ou suppression) avant l'entraînement du modèle de classification.

Notre objectif étant de faire une classification binaire sur `ArrDel15`, nous devons supprimer les lignes où cette variable est manquante (environ 2,1% des données). Il s'agit des vols annulés.

Les valeurs de temps (ex. `DepTime`, `ArrTime`) sont au format float (ex. 1345.0 pour 13h45), ce qui peut nécessiter une conversion en format horaire standard pour une meilleure interprétation.

`DayOfWeek` est codé de 1 (lundi) à 7 (dimanche), ce qui est utile pour capturer les variations hebdomadaires des retards.
`DayofMonth` et `Month` sont également présents, permettant d'analyser les tendances saisonnières. Ils sont codés avec des entiers.

D'après l'exploration des données, nous pouvons classifier les variables ainsi :

**Variables Catégorielles :**
*   **Identifiants & Codes :** `Airline`, `Origin`, `Dest`, `Marketing_Airline_Network`, `Operating_Airline`, `Tail_Number`, `IATA_Code_Marketing_Airline`, etc.
*   **Temporelles (discrètes) :** `Year`, `Quarter`, `Month`, `DayofMonth`, `DayOfWeek`.
*   **Indicateurs binaires :** `Cancelled`, `Diverted`, `DepDel15`, `ArrDel15` (Target).
*   **Groupements :** `DepartureDelayGroups`, `ArrivalDelayGroups`, `DistanceGroup`, `DepTimeBlk`, `ArrTimeBlk`.

**Variables Continues (Numériques) :**
*   **Temps de vol & Retards :** `DepDelay`, `DepDelayMinutes`, `ArrDelay`, `ArrDelayMinutes`, `AirTime`, `ActualElapsedTime`, `CRSElapsedTime`, `TaxiIn`, `TaxiOut`.
*   **Distance :** `Distance`.
*   **Horaires (à convertir) :** `DepTime`, `ArrTime`, `CRSDepTime`, `CRSArrTime`, `WheelsOff`, `WheelsOn`.

**Note :** Certaines variables comme `OriginAirportID` ou `OriginStateFips` sont stockées comme des entiers (`int64`) mais sont conceptuellement des variables **catégorielles** (identifiants).


## 3. Nettoyage des données

Nous avons défini notre problème comme une classification binaire stricte. Les vols annulés ou déroutés ne sont pas pertinents pour prédire un retard à l'arrivée. Nous les supprimons donc.

En partant d'un échantillon brut de 100 000 vols, notre nettoyage (suppression des annulations et des valeurs manquantes) nous laisse avec 97 896 vols exploitables. C'est une perte de données minime (~2%) qui garantit la qualité de notre apprentissage.

In [ ]:
# 1. Nettoyage (Drop) : Suppression des colonnes "Post-Décollage" (Data Leakage)
leakage_cols = [
    'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrivalDelayGroups', 'ArrTimeBlk',
    'ActualElapsedTime', 'AirTime', 'WheelsOn', 'TaxiIn',
    'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'WheelsOff', 'TaxiOut',
    'Tail_Number'
]

# On garde ArrDel15 car c'est la target, mais on supprime les autres fuites
cols_to_drop = [col for col in leakage_cols if col in df.columns and col != 'ArrDel15']

df_clean = df[(df['Cancelled'] == False) & (df['Diverted'] == False)].drop(columns=cols_to_drop, errors='ignore').copy()

# Supprimer les lignes avec ArrDel15 manquant (vols sans info de retard)
df_clean = df_clean.dropna(subset=['ArrDel15'])

print(f"Dataset nettoyé : {len(df_clean):,} vols")

In [ ]:
df_clean.info()

Désormais il n'y a plus de données nulles

## 4. Séparation train/test et distribution de la cible

In [ ]:
from sklearn.model_selection import train_test_split

# Créer le jeu de test (20%) et le mettre de côté
df_train, df_test = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['ArrDel15']  # Préserve la proportion des classes
)

print(f"Training set: {len(df_train)} samples")
print(f"Test set: {len(df_test)} samples")
print(f"\nTest set - Distribution ArrDel15:\n{df_test['ArrDel15'].value_counts(normalize=True)}")

## 5. Visualisation avancée (train)

In [ ]:

# 1. Distribution de la cible (ArrDel15)
plt.figure(figsize=(6, 4))
sns.countplot(x=target, data=df_train)
plt.title('Distribution des Retards > 15 min (Train set)')
plt.xlabel('Retard (0 = Non, 1 = Oui)')
plt.ylabel('Nombre de vols')
plt.show()

ratio_retard = df_train[target].mean()
print(f"Taux de retard global : {ratio_retard:.2%}")

In [ ]:
# 2. Distribution des variables numériques clés
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_train['Distance'], bins=50, ax=axes[1])
axes[1].set_title('Distribution de la Distance')

plt.show()

In [ ]:
# 3. Analyse des Compagnies Aériennes

# Taux de retard moyen par compagnie
airline_delay = df_train.groupby('Airline')[target].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(y=airline_delay.index, x=airline_delay.values, hue=airline_delay.index, palette='coolwarm', legend=False)
plt.title('Taux de retard moyen par Compagnie Aérienne')
plt.xlabel('Proportion de retards')
plt.show()

In [ ]:
# 3.5 Analyse des Aéroports (Origin, Destination)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 aéroports de départ
top_origin = df_train['Origin'].value_counts().nlargest(15)
sns.barplot(x=top_origin.values, y=top_origin.index, ax=axes[0], palette='Blues_d', hue=top_origin.index, legend=False)
axes[0].set_title('Top 15 Aéroports de Départ (Origin)')
axes[0].set_xlabel('Nombre de vols')

# Top 15 aéroports d'arrivée
top_dest = df_train['Dest'].value_counts().nlargest(15)
sns.barplot(x=top_dest.values, y=top_dest.index, ax=axes[1], palette='Greens_d', hue=top_dest.index, legend=False)
axes[1].set_title('Top 15 Aéroports d\'Arrivée (Dest)')
axes[1].set_xlabel('Nombre de vols')

plt.tight_layout()
plt.show()

# Aéroports totaux
print(f"Nombre total d'aéroports de départ : {df_train['Origin'].nunique()}")
print(f"Nombre total d'aéroports d'arrivée : {df_train['Dest'].nunique()}")

Il y a 373 aéroports uniques dans le jeu de données d'entraînement. Sachant que Dest et Origin sont distincts, cela se traduit par 746 catégories uniques pour les features catégorielles `Origin` et `Dest`.

In [ ]:
# 4. Analyse de la Distance
plt.figure(figsize=(10, 6))
sns.boxplot(x=target, y='Distance', data=df_train, hue=target, palette=['green', 'red'], legend=False)
plt.xticks([0, 1], ['À l\'heure', 'En retard'])
plt.title('Distribution de la distance par statut de retard')
plt.xlabel('Statut du vol')
plt.ylabel('Distance (miles)')
plt.show()

# Relation entre distance et retard (par groupe)
plt.figure(figsize=(10, 6))
distance_delay = df_train.groupby('DistanceGroup')[target].mean().sort_index()
sns.lineplot(x=distance_delay.index, y=distance_delay.values, marker='o', color='coral', linewidth=2)
plt.title('Taux de retard en fonction du groupe de distance')
plt.xlabel('Groupe de distance')
plt.ylabel('Proportion de retards')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 5. Facteurs Temporels
# Distribution des retards par plage horaire de départ
plt.figure(figsize=(12, 6))
delay_by_time = df_train.groupby('DepTimeBlk')[target].mean().sort_values(ascending=False)
sns.barplot(x=delay_by_time.index, y=delay_by_time.values, hue=delay_by_time.index, palette='viridis', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title('Taux de retard par plage horaire de départ')
plt.xlabel('Plage horaire')
plt.ylabel('Proportion de retards')
plt.tight_layout()
plt.show()

# Distribution des retards par mois
plt.figure(figsize=(10, 5))
month_labels = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun', 'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']
delay_by_month = df_train.groupby('Month')[target].mean().sort_index()
sns.barplot(x=delay_by_month.index, y=delay_by_month.values, palette='Oranges_d', hue=delay_by_month.index, legend=False)
plt.xticks(range(12), month_labels)
plt.title('Taux de retard par mois')
plt.xlabel('Mois')
plt.ylabel('Proportion de retards')
plt.show()

In [ ]:
# 6. Matrice de corrélation
# Matrice de corrélation étendue avec features temporelles et opérationnelles
corr_features = [
    # Temporel
    'Month', 'Quarter', 'DayOfWeek', 'DayofMonth',
    # Horaire
    'CRSDepTime',
    # Distance/Durée
    'Distance', 'CRSElapsedTime', 'DistanceGroup',
    # Target
    target
]

corr_data = df_train[corr_features].dropna()

plt.figure(figsize=(14, 10))
correlation_matrix = corr_data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            linewidths=0.5, cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de corrélation étendue - Features temporelles et opérationnelles', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## 6. Exploration non supervisée (PCA & clustering)

### PCA (Analyse en composantes principales)

In [ ]:
# Préparation des données pour l'apprentissage non supervisé
subset_size = 20000
df_pca = df_train.sample(n=subset_size, random_state=42) if len(df_train) > subset_size else df_train.copy()

features_pca = ['Distance', 'CRSElapsedTime', 'CRSDepTime', 'CRSArrTime']
# Imputation simple pour la viz (remplacer NaN par 0 ou médiane)
X_pca = df_pca[features_pca].fillna(0) 

# Standardisation
scaler_viz = StandardScaler()
X_scaled = scaler_viz.fit_transform(X_pca)

# PCA
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(components[:, 0], components[:, 1], 
            c=df_pca['ArrDel15'], cmap='coolwarm', alpha=0.5, s=15)
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.title('PCA - Projection des Vols (Rouge = Retard)')
plt.legend(handles=scatter.legend_elements()[0], labels=['À l\'heure', 'En Retard'])
plt.show()

On ne peut pas séparer linéairement, les vols à l'heure des vols en retard. Ils faut utiliser des modèles plus robustes.

In [ ]:
## 6. Analyse Clustering : Détermination du k optimal (Méthode du Coude)

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Sélection des features pertinentes pour le clustering
# On utilise des variables numériques qui définissent les caractéristiques du vol
features_cluster = ['CRSDepTime', 'CRSElapsedTime', 'Distance']
X_cluster = df_train[features_cluster].copy()

# 2. Standardisation des données
# K-Means est sensible aux échelles (la distance prime), il faut donc normaliser
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# 3. Calcul de l'inertie pour différentes valeurs de k (de 1 à 10)
inertia = []
k_range = range(1, 11)

print("Calcul en cours pour les clusters...")
for k in k_range:
    # n_init=10 permet de lancer l'algo 10 fois pour éviter les optimums locaux
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertia.append(kmeans.inertia_)

# 4. Tracé de la courbe (Elbow Curve)
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o', linestyle='--', color='b')
plt.title('Méthode Elbow')
plt.xlabel('Nombre de clusters (k)')
plt.ylabel('Inertie (WCSS)')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:

# Grâce à la méthode Elbow on constate que le kmeans optimal est de 3
# Clustering K-Means
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df_pca['Cluster'] = clusters

# Visualisation des clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=df_pca['Cluster'], palette='viridis', alpha=0.6)
plt.title('K-Means Clustering (3 clusters)')
plt.show()

# Analyse rapide des clusters
print(df_pca.groupby('Cluster')[['ArrDel15', 'CRSElapsedTime', 'Distance', 'CRSDepTime', 'Airline']].mean())

Ici on identifie 3 clusters :
- *0* : Vols courts courriers à l'heure, taux de retards de ~13%
- *1* : Vols courts couriers à risque, taux de retards de ~22%
- *2* : Vols longs courriers, taux de retard de ~17%

## 7. Préparation des features et preprocessing

In [ ]:
# Définir les features et la target
features = [
    # Temporelles
    'Month', 'DayOfWeek', 'DayofMonth', 'Quarter',
    # Horaire
    'CRSDepTime', 'CRSArrTime', 'CRSElapsedTime',
    # Catégorielles
    'Airline', 'Origin', 'Dest',
    # Distance
    'Distance'
]
target = 'ArrDel15'

# Séparation X/y pour train et test
X_train = df_train[features].copy()
y_train = df_train[target].copy()

X_test = df_test[features].copy()
y_test = df_test[target].copy()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"{len(features)} features sélectionnées")

In [ ]:
# Identification des colonnes numériques et catégorielles
categorical_features = ['Airline', 'Origin', 'Dest']
numerical_features = ['Month', 'DayOfWeek', 'DayofMonth', 'Quarter',
                      'CRSDepTime', 'CRSArrTime', 'CRSElapsedTime', 'Distance',
                      'IsWeekend', 'IsHolidayMonth', 'DepHour', 'ArrHour' ]

# Création de nouvelles features basées sur l'analyse EDA
for dataset in [df, df_train, df_test, X_train, X_test]:
    dataset['IsWeekend'] = (dataset['DayOfWeek'] >= 6).astype(int)
    dataset['IsHolidayMonth'] = dataset['Month'].isin([6, 7, 12]).astype(int)
    dataset['DepHour'] = dataset['CRSDepTime'] // 100
    dataset['ArrHour'] = dataset['CRSArrTime'] // 100


# Pipeline pour les features numériques : imputation + standardisation
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline pour les features catégorielles : imputation + encodage one-hot
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer pour appliquer les pipelines
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features),
])

preprocessor

## 8. Modèles de référence

### 8.1 Baseline : régression logistique

In [ ]:
from sklearn.model_selection import cross_val_score
import time

# Créer un pipeline complet : preprocessing + modèle
logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1))
])

logistic_pipeline

In [ ]:
y_train_pred_lp = cross_val_predict(logistic_pipeline, X_train, y_train, cv=3)

plot_confusion_matrix(y_train, y_train_pred_lp, "Logistic Regression (sans class_weight)")

Les résultats sur le jeu d'entraînement confirment nos craintes sur les modèles linéaires simples :

Accuracy : 82,81% (Semble bon, mais trompeur car proche du taux de vols à l'heure).

Recall (Rappel) : 1,59% (Catastrophique).

ROC-AUC : 0,68. Ce modèle est inutile opérationnellement : il ne détecte quasiment aucun retard (seulement 1,6% des vrais retards sont trouvés). Il se contente de prédire "À l'heure" presque tout le temps.

In [ ]:
logistic_pipeline_2 = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    ))
])
logistic_pipeline_2

In [ ]:
y_train_pred_lp2 = cross_val_predict(logistic_pipeline_2, X_train, y_train, cv=3)

plot_confusion_matrix(y_train, y_train_pred_lp2, "Logistic Regression (avec class_weight)")

Notre F1-Score a augmenté (passant de 2,88% à 35,59%), ce qui prouve que le modèle a enfin cessé d'ignorer la classe minoritaire et détecte désormais 60% des retards réels.

Mais la Précision à chuté de 2%. Pour capturer ces retards, le modèle génère désormais énormément de fausses alarmes. Cela illustre le compromis entre Précision et Rappel.

### 8.2 Random Forest

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline

In [ ]:
y_train_pred_rf = cross_val_predict(rf_pipeline, X_train, y_train, cv=3)

plot_confusion_matrix(y_train, y_train_pred_rf, "Logistic Regression (avec class_weight)")

### 8.3 Comparaison sur le train

In [ ]:
conf_mx_lp2 = confusion_matrix(y_train, y_train_pred_lp2)
conf_mx_rf = confusion_matrix(y_train, y_train_pred_rf)

# Nous avons besoin de fit pour obtenir les roc-auc
logistic_pipeline_2.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

y_train_proba_lp2 = logistic_pipeline_2.predict_proba(X_train)[:, 1]
y_train_proba_rf = rf_pipeline.predict_proba(X_train)[:, 1]

# Calcul des métriques
metrics = {
    'Modèle': ['Logistic Regression (with class_weight)', 'Random Forest'],
    'Accuracy': [accuracy_score(y_train, y_train_pred_lp2), accuracy_score(y_train, y_train_pred_rf)],
    'Precision': [precision_score(y_train, y_train_pred_lp2), precision_score(y_train, y_train_pred_rf)],
    'Recall': [recall_score(y_train, y_train_pred_lp2), recall_score(y_train, y_train_pred_rf)],
    'F1-Score': [f1_score(y_train, y_train_pred_lp2), f1_score(y_train, y_train_pred_rf)],
    'ROC-AUC': [roc_auc_score(y_train, y_train_proba_lp2),
                roc_auc_score(y_train, y_train_proba_rf)]
}

comparison_df = pd.DataFrame(metrics)
print("RÉSULTATS SUR LE TRAIN SET")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)
# Meilleur modèle
best_idx = comparison_df['F1-Score'].idxmax()
best_model = comparison_df.loc[best_idx, 'Modèle']
print(f"\n🏆 Meilleur modèle (Train Set) : {best_model}")
print(f"   F1-Score : {comparison_df.loc[best_idx, 'F1-Score']:.4f}")

### 8.4 Évaluation sur le test

In [ ]:
# Logistic Regression (with class_weight)
y_test_pred_lr = logistic_pipeline_2.predict(X_test)
y_test_proba_lr = logistic_pipeline_2.predict_proba(X_test)
test_accuracy_lr = accuracy_score(y_test, y_test_pred_lr)
test_precision_lr = precision_score(y_test, y_test_pred_lr)
test_recall_lr = recall_score(y_test, y_test_pred_lr)
test_f1_lr = f1_score(y_test, y_test_pred_lr)
test_roc_auc_lr = roc_auc_score(y_test, y_test_proba_lr[:, 1])

# Random Forest
y_test_pred_rf = rf_pipeline.predict(X_test)
y_test_proba_rf = rf_pipeline.predict_proba(X_test)
test_accuracy_rf = accuracy_score(y_test, y_test_pred_rf)
test_precision_rf = precision_score(y_test, y_test_pred_rf)
test_recall_rf = recall_score(y_test, y_test_pred_rf)
test_f1_rf = f1_score(y_test, y_test_pred_rf)
test_roc_auc_rf = roc_auc_score(y_test, y_test_proba_rf[:, 1])

# Comparaison Test Set
test_comparison_df = pd.DataFrame({
    'Modèle': ['Logistic Regression (with class_weight)', 'Random Forest'],
    'Accuracy': [test_accuracy_lr, test_accuracy_rf],
    'Precision': [test_precision_lr, test_precision_rf],
    'Recall': [test_recall_lr, test_recall_rf],
    'F1-Score': [test_f1_lr, test_f1_rf],
    'ROC-AUC': [test_roc_auc_lr, test_roc_auc_rf]
})

print("RÉSULTATS SUR LE TEST SET")
print("="*70)
print(test_comparison_df.to_string(index=False))
print("="*70)


Résumé des performances test : le Random Forest est légèrement meilleur en accuracy et F1, la régression logistique garde un rappel plus élevé (plus d'alertes).

## 9. Modèles avancés et optimisation

### 9.1 Ensembles : boosting et stacking

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, StackingClassifier
from sklearn.ensemble import RandomForestClassifier

# --- Appliquer le préprocesseur (une seule fois) ---
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# --- HistGradientBoosting sur les données prétraitées ---
hgb_model = HistGradientBoostingClassifier(random_state=42, class_weight='balanced')
hgb_model.fit(X_train_preprocessed, y_train)

y_test_pred_hgb = hgb_model.predict(X_test_preprocessed)
y_test_proba_hgb = hgb_model.predict_proba(X_test_preprocessed)[:, 1]

print("HGB F1-Score : {:.4f}".format(f1_score(y_test, y_test_pred_hgb)))
print("HGB ROC-AUC  : {:.4f}".format(roc_auc_score(y_test, y_test_proba_hgb)))

print("\nEntraînement du Stacking Classifier...")

estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, class_weight='balanced', n_jobs=-1)),
    ('hgb', HistGradientBoostingClassifier(random_state=42, class_weight='balanced'))
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=3,
    n_jobs=-1
)

stacking_clf.fit(X_train_preprocessed, y_train)

y_test_pred_stack = stacking_clf.predict(X_test_preprocessed)
y_test_proba_stack = stacking_clf.predict_proba(X_test_preprocessed)[:, 1]

results = {
    'Modèle': ['Random Forest', 'HistGradientBoosting', 'Stacking'],
    'F1-Score': [
        f1_score(y_test, y_test_pred_rf),
        f1_score(y_test, y_test_pred_hgb),
        f1_score(y_test, y_test_pred_stack)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_test_proba_rf[:, 1]),
        roc_auc_score(y_test, y_test_proba_hgb),
        roc_auc_score(y_test, y_test_proba_stack)
    ],
    'Accuracy': [
        accuracy_score(y_test, y_test_pred_rf),
        accuracy_score(y_test, y_test_pred_hgb),
        accuracy_score(y_test, y_test_pred_stack)
    ]
}

df_results = pd.DataFrame(results)
print("\nCLASSEMENT FINAL DES MODÈLES")
print("="*60)
print(df_results.sort_values(by='F1-Score', ascending=False).to_string(index=False))
print("="*60)


### 9.2 Recherche d'hyperparamètres

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import joblib
import os

# Hyperparameter search kept minimal for speed
param_dist = {
    "n_estimators": randint(50, 250),
    "max_depth": randint(5, 40),
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 6),
}

search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight="balanced"),
    param_distributions=param_dist,
    n_iter=50,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    refit=True,
    verbose=1,
)

# Fit on a capped sample for quicker tuning
sample_size = min(len(y_train), 50000)
print(f"RandomizedSearch on {sample_size} samples...")
search.fit(X_train_preprocessed[:sample_size], y_train[:sample_size])

best_rf_model = search.best_estimator_
print(f"Best params: {search.best_params_}")
print(f"Best CV ROC-AUC: {search.best_score_:.4f}")

# Retrain on full preprocessed train set and persist
best_rf_model.fit(X_train_preprocessed, y_train)
os.makedirs("models", exist_ok=True)
joblib.dump({"preprocessor": preprocessor, "model": best_rf_model}, "models/best_rf_pipeline.joblib")
print("Saved optimized model to models/best_rf_pipeline.joblib")


### 9.4 Évaluation finale du modèle optimisé

In [ ]:
# Évaluation simple du meilleur modèle

y_test_pred_best = best_rf_model.predict(X_test_preprocessed)
y_test_proba_best = best_rf_model.predict_proba(X_test_preprocessed)[:, 1]

print("--- Rapport de classification (Test) ---")
print(classification_report(y_test, y_test_pred_best, digits=4))

metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred_best),
    "Precision": precision_score(y_test, y_test_pred_best, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred_best, zero_division=0),
    "F1-Score": f1_score(y_test, y_test_pred_best),
    "ROC-AUC": roc_auc_score(y_test, y_test_proba_best),
}

print("\nMétriques clés (Test):")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

# Matrice de confusion concise
plot_confusion_matrix(y_test, y_test_pred_best, "Optimized Random Forest")

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, y_test_proba_best)
roc_auc_val = auc(fpr, tpr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Optimized Random Forest')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()


## Conclusion

Les résultats restent mitigés : malgré l'optimisation, les métriques (Recall/Precision/F1/ROC-AUC) ne permettent pas encore une décision fiable. Il faudra enrichir les features (ex. météo, conditions opérationnelles), calibrer les probabilités, ajuster le seuil métier et tester d'autres algorithmes (XGBoost/LightGBM) pour viser un modèle plus robuste.

## 10. Feature Importance Analysis

In [ ]:
import matplotlib.pyplot as plt

# 1. Préparation des données (on suppose que names et importances sont extraits)
df_imp = pd.DataFrame({'Feature': names, 'Importance': importances}).sort_values('Importance', ascending=False).head(20)

# 2. Plot
plt.figure(figsize=(10, 8))
# On capture directement le BarContainer ici
bars = plt.barh(df_imp['Feature'], df_imp['Importance'], color=plt.cm.viridis(np.linspace(0.8, 0.2, 20)))

# 3. Customisation
plt.gca().invert_yaxis() # Plus important en haut
plt.bar_label(bars, fmt='%.4f', padding=5) # 'bars' est ici un BarContainer valide
plt.title('Top 20 Features - Random Forest', fontweight='bold')
plt.xlabel('Importance')

plt.tight_layout()
plt.show()